# IPS Discovery Analysis Pipeline

End-to-end notebook that turns discovery-session notes and worksheets into
**categorized challenges** and **expectations**.

| Step | What it does | Key outputs |
|------|----------------|-------------|
| **1** | Split meeting notes into focus-group `.docx` files | `output/docx_sections/` |
| **2** | Parse discovery worksheets into a flat table | `output/worksheets.csv` |
| **3** | Extract pain points & future capabilities from both sources | `challenges.csv`, `expectations.csv` |
| **4** | Clean text, score sentiment, realign mislabeled rows | `clean_*.csv` |
| **5** | Hybrid keyword + embedding categorization & clustering | `categorized_*.csv` |
| **6** | Interactive charts for review | plots in-notebook |

**How to run:** execute cells top-to-bottom. Later sections reuse in-memory
dataframes when available and only fall back to CSV if you restart mid-way.


## 0. Setup

Install dependencies once, then load shared imports, paths, and constants used
by every later section.


In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Shared imports & configuration
from __future__ import annotations

import os
import re
import zipfile
from pathlib import Path

import hdbscan
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import umap
from docx import Document
from IPython.display import Markdown, display
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# Optional interactive heatmap (fails gracefully outside Jupyter widgets)
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

# --- Paths ---
NOTES_PATH = Path("./data/notes.docx")
WORKSHEETS_DIR = Path("./data/worksheets")

OUTPUT_DIR = Path("./output")          # clean / final products
RAW_DIR = OUTPUT_DIR / "raw"           # raw extracts
SECTIONS_DIR = RAW_DIR / "docx_sections"

WORKSHEETS_CSV = RAW_DIR / "worksheets.csv"
CHALLENGES_CSV = RAW_DIR / "challenges.csv"
EXPECTATIONS_CSV = RAW_DIR / "expectations.csv"

CLEAN_CHALLENGES_CSV = OUTPUT_DIR / "clean_challenges.csv"
CLEAN_EXPECTATIONS_CSV = OUTPUT_DIR / "clean_expectations.csv"
CATEGORIZED_CHALLENGES_CSV = OUTPUT_DIR / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = OUTPUT_DIR / "categorized_expectations.csv"
CATEGORY_SUMMARY_CSV = OUTPUT_DIR / "category_summary.csv"

for path in (OUTPUT_DIR, RAW_DIR, SECTIONS_DIR):
    path.mkdir(parents=True, exist_ok=True)

# --- Models / thresholds ---
SENTIMENT_MODEL = "MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33"
# Workplace / discovery framing (not tweet emotion)
SENTIMENT_HYPOTHESIS = "This workplace note indicates {}."
SENTIMENT_LABELS = [
    "process friction that needs fixing",
    "a desired future capability",
    "neither friction nor a request",
]
SENTIMENT_LABEL_MAP = {
    "process friction that needs fixing": "negative",
    "a desired future capability": "positive",
    "neither friction nor a request": "neutral",
}

EMBEDDING_MODEL = "all-mpnet-base-v2"
MIN_WORDS = 4
TITLE_KEYWORD_MIN = 4
BODY_KEYWORD_MIN = 4
BODY_KEYWORD_LOW_MIN = 2
SEMANTIC_MIN = 0.35

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"

print("Setup complete. Raw →", RAW_DIR.resolve(), "| Clean →", OUTPUT_DIR.resolve())


Setup complete. Outputs → /Users/lilscott/Code/IPS/simple/output


## 1. Split Discovery Notes by Focus Group

Meeting notes (`data/notes.docx`) use **Heading 3** for each discovery session.
This step:

1. Detects every Heading 3 block
2. Collapses session titles into a short **focus-group** label (drops
   “Discovery Session”, group numbers, etc.)
3. Merges sessions that share a label into one `.docx` under
   `output/docx_sections/`

> Later steps read those files for Heading 4 sections
> (*Pain Points & Challenges* / *Future Capabilities*).


In [3]:
def simple_focus_group_name(heading_text: str, fallback: str = "section") -> str:
    """Turn a Discovery Session Heading 3 into a short shared focus-group label."""
    text = str(heading_text).replace("\xa0", " ")
    text = re.sub(r"(?i)\bdiscovery\s+session\b", "", text)
    parts = [p.strip() for p in re.split(r"\s*\|\s*", text) if p.strip()]

    cleaned = []
    for part in parts:
        part = part.replace("/", "")
        part = re.sub(r"(?i)\s+group\s*\d*$", "", part).strip()
        part = re.sub(r"\s+\d+$", "", part).strip()
        part = re.sub(r"\s{2,}", " ", part).strip(" .-_")
        if part:
            cleaned.append(part)

    name = " ".join(cleaned)
    name = re.sub(r"\b(\w+)\s+\1\b", r"\1", name, flags=re.I)
    name = re.sub(r"\s{2,}", " ", name).strip(" .-_")
    name = "".join(ch for ch in name if ch not in '<>:"/\\|?*').strip()
    name = re.sub(r"\s{2,}", " ", name).strip(" .-_")
    return name or fallback


def _copy_paragraph(src_para, dst_doc):
    """Copy a paragraph with basic run formatting (needed for Heading styles)."""
    new_para = dst_doc.add_paragraph()
    try:
        if src_para.style is not None:
            new_para.style = src_para.style.name
    except Exception:
        pass
    for run in src_para.runs:
        new_run = new_para.add_run(run.text)
        for attr in ("bold", "italic", "underline"):
            try:
                setattr(new_run, attr, getattr(run, attr))
            except Exception:
                pass
        try:
            if run.font is not None:
                if run.font.name:
                    new_run.font.name = run.font.name
                if run.font.size:
                    new_run.font.size = run.font.size
        except Exception:
            pass
    return new_para


def split_docx_by_heading(input_path: Path | str, output_dir: Path | str) -> dict[str, int]:
    """Split notes.docx on Heading 3; merge same focus-group into one file."""
    input_path, output_dir = Path(input_path), Path(output_dir)
    if not input_path.exists():
        raise FileNotFoundError(f"Missing notes file: {input_path}")

    doc = Document(input_path)
    sections: list[tuple] = []
    current = None

    for para in doc.paragraphs:
        style_name = para.style.name if para.style else ""
        if style_name.startswith("Heading 3"):
            if current is not None:
                sections.append(current)
            current = (para, [])
        elif current is not None:
            current[1].append(para)
    if current is not None:
        sections.append(current)

    # Clear stale section files only
    if output_dir.is_dir():
        for existing in output_dir.glob("*.docx"):
            existing.unlink()
    output_dir.mkdir(parents=True, exist_ok=True)

    written: dict[str, int] = {}
    for index, (heading_para, content_paras) in enumerate(sections, start=1):
        heading_text = heading_para.text if heading_para is not None else f"section_{index}"
        focus_group = simple_focus_group_name(heading_text, fallback=f"section_{index}")
        out_path = output_dir / f"{focus_group}.docx"

        section_doc = Document(out_path) if focus_group in written else Document()
        _copy_paragraph(heading_para, section_doc)
        for paragraph in content_paras:
            _copy_paragraph(paragraph, section_doc)
        section_doc.save(out_path)

        written[focus_group] = written.get(focus_group, 0) + 1
        action = "Merged" if written[focus_group] > 1 else "Saved"
        print(f"{action}: {out_path.name}")

    return written


counts = split_docx_by_heading(NOTES_PATH, SECTIONS_DIR)
print(f"\n{len(counts)} focus-group files in {SECTIONS_DIR}")


Saved: DOCE Admin Aides.docx
Saved: DOCE Building Inspectors.docx
Merged: DOCE Admin Aides.docx
Saved: DOCE Supervisors.docx
Saved: DOCE Housing Inspectors.docx
Saved: DOCE CommercialPermitElectrical Inspectors.docx
Saved: DOCE Fire Prevention Bureau.docx
Saved: DOCE Central Permit Office.docx
Saved: DOCE Zoning.docx
Saved: DOCE Office Staff.docx
Saved: DOCE CPO Coordinators.docx
Saved: Law.docx
Saved: Assessment.docx
Saved: BAA Supervisors & Admin Aide.docx
Saved: NBD Neighborhood Planning.docx
Saved: CPC.docx
Saved: NBD Data Team.docx
Saved: BAA Clerks & Admin Aides.docx
Saved: BAA ALJs.docx
Saved: IT.docx
Saved: NBD Kate.docx
Saved: BAA Ops.docx
Saved: NBD Internal.docx

22 focus-group files in output/docx_sections


## 2. Extract Worksheet Responses

Each worksheet under `data/worksheets/` is a Word table pack (role, process,
pain points, technology). This step maps messy filenames onto the **same
focus-group labels** used in Section 1, then flattens every completed section
into `output/worksheets.csv`.


In [4]:
# Filename → shared focus-group labels (aligns with Section 1)
WORKSHEET_FOCUS_GROUPS = {
    "baa supervisors focus group": "BAA Supervisors & Admin Aide",
    "building inspector - round 1 doce": "DOCE Building Inspectors",
    "cpc focus group": "CPC",
    "cpo central permit office": "DOCE Central Permit Office",
    "cpo co-ordinator focus group": "DOCE CPO Coordinators",
    "cpo coordinator focus group": "DOCE CPO Coordinators",
    "doce admin aide": "DOCE Admin Aides",
    "fire department focus group": "DOCE Fire Prevention Bureau",
    "housing inspectors - round 1 doce": "DOCE Housing Inspectors",
    "law": "Law",
    "nbd data team worksheet response": "NBD Data Team",
    "nbd data team": "NBD Data Team",
    "nbd focus group": "NBD Neighborhood Planning",
    "office manager - round 1 doce": "DOCE Office Staff",
    "perm_com_elec inspectors - round 1 doce": "DOCE CommercialPermitElectrical Inspectors",
    "round 1 discover worksheet response zoning": "DOCE Zoning",
    "supervisors - round 1 doce": "DOCE Supervisors",
}

DATA_COLUMNS = [
    "role", "process", "pain_point", "technology_likes",
    "technology_dislikes", "tools", "expectations",
]

LIKES_HEADER = "what features do you like about ips"
TOOLS_HEADER = "are there other tools or systems"
EXPECTATIONS_HEADER = "what features or capabilities do you wish ips had"


def dedupe_row_cells(cells: list[str]) -> list[str]:
    """Collapse duplicate cell text caused by merged Word table cells."""
    result = []
    for cell in cells:
        text = cell.strip()
        if not result or text != result[-1]:
            result.append(text)
    return result


def normalize_text(text: str) -> str:
    return text.strip().lower().replace("\u2019", "'").replace("\xa0", " ")


def get_table_type(table) -> str | None:
    if not table.rows:
        return None
    first = normalize_text(table.rows[0].cells[0].text)
    for key in ("role", "process", "pain", "technology"):
        if first.startswith(key):
            return "pain_point" if key == "pain" else key
    return None


def extract_role(table) -> str:
    return table.rows[1].cells[0].text.strip() if len(table.rows) >= 2 else ""


def extract_multiline_table(table, start_row: int = 2) -> str:
    lines = []
    for row in table.rows[start_row:]:
        cells = dedupe_row_cells([cell.text for cell in row.cells])
        if any(cells):
            lines.append(" | ".join(cells))
    return "\n".join(lines)


def extract_technology_sections(table) -> dict[str, str]:
    likes, dislikes, tools, expectations = [], [], [], []
    section = None

    for row in table.rows:
        cells = dedupe_row_cells([cell.text for cell in row.cells])
        if not any(c.strip() for c in cells):
            continue

        row_text = normalize_text(" ".join(cells))
        first_cell = normalize_text(cells[0])

        if first_cell.startswith("technology"):
            continue
        if LIKES_HEADER in row_text:
            section = "likes_dislikes"
            continue
        if TOOLS_HEADER in row_text:
            section = "tools_pending"
            continue
        if first_cell == "tool" and "used for" in row_text:
            section = "tools"
            continue
        if EXPECTATIONS_HEADER in row_text:
            section = "expectations"
            continue

        if section == "likes_dislikes":
            if cells[0].strip():
                likes.append(cells[0].strip())
            if len(cells) > 1 and cells[1].strip():
                dislikes.append(cells[1].strip())
        elif section == "tools":
            tools.append(" | ".join(c.strip() for c in cells if c.strip()))
        elif section == "expectations":
            expectations.append(" | ".join(c.strip() for c in cells if c.strip()))

    return {
        "technology_likes": "\n".join(likes),
        "technology_dislikes": "\n".join(dislikes),
        "tools": "\n".join(tools),
        "expectations": "\n".join(expectations),
    }


def empty_worksheet_row(focus_group: str) -> dict:
    return {"focus_group_name": focus_group, **{c: "" for c in DATA_COLUMNS}}


def focus_group_name_from_filename(filename: str) -> str:
    stem = Path(filename).stem.replace("\xa0", " ")
    stem = re.sub(r"\s+Worksheet(?:\s+Response)?$", "", stem, flags=re.I)
    stem = re.sub(r"\s{2,}", " ", stem).strip()
    key = stem.lower()
    if key in WORKSHEET_FOCUS_GROUPS:
        return WORKSHEET_FOCUS_GROUPS[key]

    cleaned = re.sub(r"(?i)\bfocus\s+group\b", "", stem)
    cleaned = re.sub(r"(?i)\bround\s*1\b", "", cleaned)
    cleaned = re.sub(r"(?i)\bdiscover(?:y)?\b", "", cleaned)
    cleaned = re.sub(r"(?i)\bresponse\b", "", cleaned)
    cleaned = re.sub(r"\s*[-–—]\s*", " ", cleaned)
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip(" .-_")
    return cleaned or stem


def extract_worksheet_rows(doc_path: Path) -> list[dict]:
    if doc_path.name.startswith("~$"):
        return []

    doc = Document(doc_path)
    focus_group = focus_group_name_from_filename(doc_path.name)
    rows, current = [], empty_worksheet_row(focus_group)

    for table in doc.tables:
        table_type = get_table_type(table)
        if table_type == "role":
            if any(current[k] for k in DATA_COLUMNS):
                rows.append(current)
            current = empty_worksheet_row(focus_group)
            current["role"] = extract_role(table)
        elif table_type == "process":
            current["process"] = extract_multiline_table(table)
        elif table_type == "pain_point":
            current["pain_point"] = extract_multiline_table(table)
        elif table_type == "technology":
            current.update(extract_technology_sections(table))

    if any(current[k] for k in DATA_COLUMNS):
        rows.append(current)
    return rows


all_rows = []
for doc_path in sorted(WORKSHEETS_DIR.rglob("*.docx")):
    if doc_path.name.startswith("~$"):
        continue
    try:
        rows = extract_worksheet_rows(doc_path)
        all_rows.extend(rows)
        print(f"{doc_path.name}: {len(rows)} section(s)")
    except zipfile.BadZipFile:
        print(f"Skipping {doc_path.name}: not a valid docx")
    except Exception as exc:
        print(f"Error reading {doc_path.name}: {exc}")

worksheets_df = (
    pd.DataFrame(all_rows, columns=["focus_group_name", *DATA_COLUMNS])
    .replace("", pd.NA)
)
worksheets_df.to_csv(WORKSHEETS_CSV, index=False, encoding="utf-8-sig")
print(f"\nSaved {len(worksheets_df)} rows → {WORKSHEETS_CSV}")
worksheets_df.head()


BAA Supervisors Focus Group Worksheet.docx: 4 section(s)
Building Inspector - Round 1 DOCE Worksheet.docx: 5 section(s)
CPC Focus Group Worksheet.docx: 6 section(s)
CPO Central Permit Office.docx: 3 section(s)
CPO Co-ordinator Focus Group Worksheet.docx: 3 section(s)
DOCE Admin Aide.docx: 4 section(s)
Fire Department Focus Group Worksheet.docx: 3 section(s)
Housing Inspectors - Round 1 DOCE Worksheet.docx: 4 section(s)
LAW Worksheet.docx: 3 section(s)
NBD Data Team Worksheet Response.docx: 3 section(s)
NBD Focus Group Worksheet.docx: 3 section(s)
Office Manager - Round 1 DOCE Worksheet.docx: 3 section(s)
Perm_Com_Elec Inspectors - Round 1 DOCE Worksheet.docx: 6 section(s)
Round 1 Discover Worksheet Response Zoning.docx: 7 section(s)
Supervisors - Round 1 DOCE Worksheet.docx: 7 section(s)

Saved 64 rows → output/worksheets.csv


,focus_group_name,role,process,pain_point,technology_likes,technology_dislikes,tools,expectations
0,BAA Supervisors & Admin Aide,Director / Chief ALJ for BAA,Review cases when escalated | Supervisor/ staf...,Manual ticketing process | Automate tickets\nO...,NaN,NaN,BAA email | Constituent emails | Duration of B...,NaN
1,BAA Supervisors & Admin Aide,Assist constituents with information about the...,Looking up violations | Codes & other staff wh...,Multiple “actions” not being used | Limit to t...,Search criteria,NaN,USPS tracking | Tracking certified mail,NaN
2,BAA Supervisors & Admin Aide,BAA supervisor,"Open tickets | staff | IPS, foxit, word, outlo...",Case not linked into inputted in one case not ...,Actions with space for notes\nPayment tab rela...,Payment only described with type – not linked ...,"teams\noutlook\nNYDOS\nAS400, imagemate | Conf...",Linked cases is biggest\nCase note button on t...
3,BAA Supervisors & Admin Aide,Mainly checking status of violations\nService ...,Checking notes | | IPS\npayments | | IPS,Note taking (nothing technical) | Putting note...,Don't use it enough,NaN,NaN,NaN
4,DOCE Building Inspectors,Building: Electrical Inspectors\nCode Enforcem...,Certificate of Occupancy | Business that needs...,Updated Contact Information | Make sure we hav...,Updated tablets and software that actually works,New tablets,NaN,Sprinkler system reports & elevator reports – ...


## 3. Build Challenges & Expectations Datasets

Combine two sources into long-form tables:

| Source | Pain points | Expectations |
|--------|-------------|--------------|
| Worksheets | `pain_point` column (one line = one record) | `expectations` column |
| Discovery notes | text under Heading 4 *Challenges* | text under *Future Capabilities* |

Short records (≤ 4 words) are kept through sentiment review so neutrals can be
inspected fully; they are filtered later in Section 4 after that review.


In [5]:
PAIN_HEADING = re.compile(
    r"^(?:pain\s*points?\s*(?:and|&|/)\s*)?challenges?\s*:?\s*$", re.I
)
EXPECTATION_HEADING = re.compile(r"^future\s*capabilities?\s*:?\s*$", re.I)


def normalize_focus_group(name) -> str:
    return re.sub(r"\s+", " ", str(name).replace("\xa0", " ")).strip()


def normalize_heading_text(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text).replace("\xa0", " ")).strip().rstrip(":").strip()
    text = re.sub(r"(?i)(challenges)\1+", r"\1", text)
    text = re.sub(r"(?i)(capabilities)bilities", r"\1", text)
    return text


def heading4_section_type(text: str) -> str | None:
    normalized = normalize_heading_text(text)
    if PAIN_HEADING.match(normalized):
        return "pain"
    if EXPECTATION_HEADING.match(normalized):
        return "expectation"
    return None


def split_worksheet_records(cell_value) -> list[str]:
    if pd.isna(cell_value) or not str(cell_value).strip():
        return []
    return [
        line.strip().strip("|").strip()
        for line in str(cell_value).split("\n")
        if line.strip().strip("|").strip()
    ]


def extract_from_worksheets(worksheets) -> dict[str, list]:
    df = worksheets if isinstance(worksheets, pd.DataFrame) else pd.read_csv(worksheets)
    pain_points, expectations = [], []
    for row in df.itertuples(index=False):
        focus_group = normalize_focus_group(row.focus_group_name)
        pain_points.extend(
            (focus_group, t) for t in split_worksheet_records(getattr(row, "pain_point", None))
        )
        expectations.extend(
            (focus_group, t) for t in split_worksheet_records(getattr(row, "expectations", None))
        )
    print(f"Worksheets: {len(pain_points)} pain · {len(expectations)} expectations ({len(df)} rows)")
    return {"pain_points": pain_points, "expectations": expectations}


def extract_from_discovery_notes(discovery_notes_path: Path | str) -> dict[str, list]:
    pain_points, expectations = [], []
    for doc_path in sorted(Path(discovery_notes_path).glob("*.docx")):
        focus_group = normalize_focus_group(doc_path.stem)
        section = None
        for para in Document(doc_path).paragraphs:
            style = para.style.name if para.style else ""
            text = para.text.strip()
            if style.startswith("Heading"):
                section = heading4_section_type(text) if style.startswith("Heading 4") else None
                continue
            if not text or section is None:
                continue
            (pain_points if section == "pain" else expectations).append((focus_group, text))
    print(f"Discovery notes: {len(pain_points)} pain · {len(expectations)} expectations")
    return {"pain_points": pain_points, "expectations": expectations}


def filter_short(df: pd.DataFrame, text_col: str, min_words: int = MIN_WORDS) -> pd.DataFrame:
    return df[df[text_col].astype(str).str.split().str.len() > min_words].reset_index(drop=True)


ws_source = worksheets_df if "worksheets_df" in globals() and worksheets_df is not None else WORKSHEETS_CSV
worksheet_results = extract_from_worksheets(ws_source)
discovery_results = extract_from_discovery_notes(SECTIONS_DIR)

challenges_df = pd.DataFrame(
    worksheet_results["pain_points"] + discovery_results["pain_points"],
    columns=["focus_group", "pain_points"],
)
expectations_df = pd.DataFrame(
    worksheet_results["expectations"] + discovery_results["expectations"],
    columns=["focus_group", "expectations"],
)

print(f"Combined: {len(challenges_df)} pain · {len(expectations_df)} expectations "
      f"(short rows kept for neutral inspection)")

ws_groups = {fg for fg, _ in worksheet_results["pain_points"] + worksheet_results["expectations"]}
dn_groups = {fg for fg, _ in discovery_results["pain_points"] + discovery_results["expectations"]}
print(f"Focus groups in both sources: {sorted(ws_groups & dn_groups)}")
print(f"Worksheets only: {sorted(ws_groups - dn_groups)}")
print(f"Discovery only: {sorted(dn_groups - ws_groups)}")

challenges_df.to_csv(CHALLENGES_CSV, index=False, encoding="utf-8-sig")
expectations_df.to_csv(EXPECTATIONS_CSV, index=False, encoding="utf-8-sig")
print(f"Saved → {CHALLENGES_CSV.name}, {EXPECTATIONS_CSV.name}")
challenges_df["focus_group"].value_counts().head(10)


Worksheets: 172 pain · 57 expectations (64 rows)
Discovery notes: 863 pain · 163 expectations
Dropped ≤4-word rows: pain 1035 → 856, exp 220 → 182
Focus groups in both sources: ['BAA Supervisors & Admin Aide', 'CPC', 'DOCE Admin Aides', 'DOCE Building Inspectors', 'DOCE CPO Coordinators', 'DOCE Central Permit Office', 'DOCE CommercialPermitElectrical Inspectors', 'DOCE Fire Prevention Bureau', 'DOCE Housing Inspectors', 'DOCE Office Staff', 'DOCE Supervisors', 'DOCE Zoning', 'Law', 'NBD Neighborhood Planning']
Worksheets only: ['NBD Data Team']
Discovery only: ['Assessment', 'BAA Clerks & Admin Aides', 'BAA Ops', 'IT', 'NBD Internal', 'NBD Kate']
Saved → challenges.csv, expectations.csv


focus_group
DOCE Admin Aides                              96
DOCE CommercialPermitElectrical Inspectors    90
DOCE Supervisors                              79
DOCE Building Inspectors                      77
BAA Supervisors & Admin Aide                  69
DOCE Housing Inspectors                       62
DOCE Central Permit Office                    61
DOCE Zoning                                   57
DOCE Office Staff                             48
DOCE Fire Prevention Bureau                   36
Name: count, dtype: int64

## 4. Clean Text & Align Sentiment

Pipeline:

1. **Consolidate** one-word speaker labels into the next multi-word line
2. **Normalize** text for the classifier (letters/spaces only, lowercased)
3. **Score sentiment** with a **business-process zero-shot model**
   (`MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33`) using workplace labels:
   - *process friction that needs fixing* → negative (challenge)
   - *a desired future capability* → positive (expectation)
   - *neither friction nor a request* → neutral
4. **Realign** misfiled rows: positive “pain” → expectations, negative “expectation” → pain
5. **Inspect remaining neutrals** — chart (hover for text) + full tables
6. **Drop leftover neutrals**, then drop ≤4-word rows
7. Write `clean_challenges.csv` / `clean_expectations.csv`


In [6]:
def consolidate_one_word_with_next(
    df: pd.DataFrame, text_col: str, group_col: str = "focus_group"
) -> pd.DataFrame:
    """Merge one-word rows into the next multi-word row within the same focus group."""
    rows = df.reset_index(drop=True).to_dict("records")
    result, i = [], 0

    while i < len(rows):
        group = rows[i][group_col]
        text = "" if pd.isna(rows[i][text_col]) else str(rows[i][text_col]).strip()
        if not text:
            i += 1
            continue

        if len(text.split()) == 1:
            parts = [text]
            j = i + 1
            while j < len(rows) and rows[j][group_col] == group:
                nxt = "" if pd.isna(rows[j][text_col]) else str(rows[j][text_col]).strip()
                if not nxt:
                    j += 1
                    continue
                if len(nxt.split()) == 1:
                    parts.append(nxt)
                    j += 1
                    continue
                rows[j][text_col] = f"{' '.join(parts)} {nxt}".strip()
                i = j
                break
            else:
                result.append({group_col: group, text_col: " ".join(parts)})
                i = j
            continue

        result.append({group_col: group, text_col: text})
        i += 1

    return pd.DataFrame(result, columns=[group_col, text_col])


def clean_for_sentiment(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"[^A-Za-z\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.lower()
    )


def sentiment_summary(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    return (
        df.groupby("focus_group", dropna=False)
        .agg(
            total=(text_col, "count"),
            negative=("sentiment", lambda s: (s == "negative").sum()),
            positive=("sentiment", lambda s: (s == "positive").sum()),
            neutral=("sentiment", lambda s: (s == "neutral").sum()),
        )
        .reset_index()
        .sort_values(["total", "focus_group"], ascending=[False, True])
    )


# Load (prefer in-memory frames from Section 3)
df_pain_points = (
    challenges_df.copy() if "challenges_df" in globals() else pd.read_csv(CHALLENGES_CSV)
)
df_expectations = (
    expectations_df.copy() if "expectations_df" in globals() else pd.read_csv(EXPECTATIONS_CSV)
)

before = (len(df_pain_points), len(df_expectations))
df_pain_points = consolidate_one_word_with_next(df_pain_points, "pain_points")
df_expectations = consolidate_one_word_with_next(df_expectations, "expectations")
print(f"After consolidate: pain {before[0]} → {len(df_pain_points)}, "
      f"exp {before[1]} → {len(df_expectations)} "
      f"(≤{MIN_WORDS}-word rows still included)")

df_pain_points["processed_text"] = clean_for_sentiment(df_pain_points["pain_points"])
df_expectations["processed_text"] = clean_for_sentiment(df_expectations["expectations"])


After consolidate + short filter: pain 856 → 856, exp 182 → 182


In [7]:
# Business-process zero-shot sentiment (workplace / discovery framing).
# Example: "forms are a manual process" → process friction → negative
classifier = pipeline(
    "zero-shot-classification",
    model=SENTIMENT_MODEL,
    multi_label=False,
)

def predict_sentiment(texts: list[str], batch_size: int = 8) -> list[str]:
    """Classify each note as process friction, desired capability, or neither."""
    labels_out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        results = classifier(
            batch,
            candidate_labels=SENTIMENT_LABELS,
            hypothesis_template=SENTIMENT_HYPOTHESIS,
            truncation=True,
            max_length=512,
        )
        if isinstance(results, dict):
            results = [results]
        for result in results:
            labels_out.append(SENTIMENT_LABEL_MAP[result["labels"][0]])
    return labels_out


# Sanity check on a factual process-pain phrase
example = "forms are a manual process"
example_pred = classifier(
    example,
    candidate_labels=SENTIMENT_LABELS,
    hypothesis_template=SENTIMENT_HYPOTHESIS,
)
print(f'Example: "{example}"')
print(f"Hypothesis: {SENTIMENT_HYPOTHESIS}")
for lab, score in zip(example_pred["labels"], example_pred["scores"]):
    print(f"  {SENTIMENT_LABEL_MAP[lab]:>8s}  {score:.3f}  ← {lab}")
print(f"→ assigned: {SENTIMENT_LABEL_MAP[example_pred['labels'][0]]}")

df_pain_points["sentiment"] = predict_sentiment(df_pain_points["processed_text"].tolist())
df_expectations["sentiment"] = predict_sentiment(df_expectations["processed_text"].tolist())

print("\nPain sentiment:\n", df_pain_points["sentiment"].value_counts().to_string())
print("\nExpectation sentiment:\n", df_expectations["sentiment"].value_counts().to_string())
display(sentiment_summary(df_pain_points, "pain_points").head(10))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pain sentiment:
 sentiment
neutral     619
negative    200
positive     37

Expectation sentiment:
 sentiment
neutral     143
negative     29
positive     10


,focus_group,total,negative,positive,neutral
4,DOCE Admin Aides,96,22,4,70
8,DOCE CommercialPermitElectrical Inspectors,90,25,5,60
12,DOCE Supervisors,79,26,4,49
5,DOCE Building Inspectors,77,18,4,55
2,BAA Supervisors & Admin Aide,69,12,1,56
10,DOCE Housing Inspectors,62,9,2,51
7,DOCE Central Permit Office,61,10,5,46
13,DOCE Zoning,57,9,3,45
11,DOCE Office Staff,48,14,0,34
9,DOCE Fire Prevention Bureau,36,4,3,29


In [ ]:
# Realign misclassified rows between datasets
shared = ["focus_group", "processed_text", "sentiment"]

to_expectations = (
    df_pain_points.loc[df_pain_points["sentiment"] == "positive", shared + ["pain_points"]]
    .rename(columns={"pain_points": "expectations"})
)
to_pain = (
    df_expectations.loc[df_expectations["sentiment"] == "negative", shared + ["expectations"]]
    .rename(columns={"expectations": "pain_points"})
)

df_pain_points = pd.concat(
    [
        df_pain_points.loc[df_pain_points["sentiment"] != "positive", shared + ["pain_points"]],
        to_pain,
    ],
    ignore_index=True,
)
df_expectations = pd.concat(
    [
        df_expectations.loc[df_expectations["sentiment"] != "negative", shared + ["expectations"]],
        to_expectations,
    ],
    ignore_index=True,
)
print(f"Moved {len(to_expectations)} positive→expectations, {len(to_pain)} negative→pain")
print(f"Shapes after realign: pain {df_pain_points.shape}, expectations {df_expectations.shape}")
print("Pain sentiment:\n", df_pain_points["sentiment"].value_counts().to_string())
print("\nExpectation sentiment:\n", df_expectations["sentiment"].value_counts().to_string())


### Inspect remaining neutral records

These scored as **neither friction nor a request**. Use the **hover chart** and
**tables** to review before the next cell drops them.


In [ ]:
# Build a unified neutrals frame for chart + tables
neutral_pain = df_pain_points.loc[
    df_pain_points["sentiment"] == "neutral",
    ["focus_group", "pain_points", "processed_text"],
].assign(dataset="pain_points").rename(columns={"pain_points": "text"})

neutral_exp = df_expectations.loc[
    df_expectations["sentiment"] == "neutral",
    ["focus_group", "expectations", "processed_text"],
].assign(dataset="expectations").rename(columns={"expectations": "text"})

neutrals = pd.concat([neutral_pain, neutral_exp], ignore_index=True)
neutrals["word_count"] = neutrals["text"].astype(str).str.split().str.len()
neutrals["short"] = neutrals["word_count"] <= MIN_WORDS

print(f"Neutral pain points: {len(neutral_pain)}")
print(f"Neutral expectations: {len(neutral_exp)}")
print(f"Short (≤{MIN_WORDS} words) among neutrals: {int(neutrals['short'].sum())}")

if neutrals.empty:
    display(Markdown("_No neutral records._"))
else:
    # One point per neutral record — hover to read the text
    neutrals = neutrals.sort_values(["dataset", "focus_group"]).reset_index(drop=True)
    neutrals["idx"] = neutrals.groupby(["dataset", "focus_group"]).cumcount()

    fig_neutral = px.scatter(
        neutrals,
        x="idx",
        y="focus_group",
        color="dataset",
        symbol="short",
        hover_data={
            "text": True,
            "processed_text": True,
            "word_count": True,
            "short": True,
            "idx": False,
            "focus_group": True,
            "dataset": True,
        },
        title="Neutral records by focus group (hover a point to read the text)",
        labels={"idx": "Record # within focus group", "focus_group": "Focus group"},
    )
    fig_neutral.update_traces(marker=dict(size=10, opacity=0.85))
    fig_neutral.update_layout(height=max(420, 28 * neutrals["focus_group"].nunique()), legend_title_text="")
    fig_neutral.show()

    # Counts by focus group (hover for totals)
    count_df = (
        neutrals.groupby(["focus_group", "dataset"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )
    fig_counts = px.bar(
        count_df,
        x="count",
        y="focus_group",
        color="dataset",
        orientation="h",
        barmode="group",
        title="Neutral count by focus group",
        hover_data={"count": True, "dataset": True, "focus_group": True},
    )
    fig_counts.update_layout(height=max(360, 28 * count_df["focus_group"].nunique()))
    fig_counts.show()

    display(Markdown("#### All neutral pain points"))
    display(neutral_pain[["focus_group", "text", "processed_text"]].reset_index(drop=True))

    display(Markdown("#### All neutral expectations"))
    display(neutral_exp[["focus_group", "text", "processed_text"]].reset_index(drop=True))


In [ ]:
# Drop neutrals, then drop short rows — keep negative pain / positive expectations
before = (len(df_pain_points), len(df_expectations))
df_pain_points = df_pain_points[df_pain_points["sentiment"] == "negative"].reset_index(drop=True)
df_expectations = df_expectations[df_expectations["sentiment"] == "positive"].reset_index(drop=True)
print(f"Dropped neutrals: pain {before[0]} → {len(df_pain_points)}, "
      f"exp {before[1]} → {len(df_expectations)}")

before_short = (len(df_pain_points), len(df_expectations))
df_pain_points = filter_short(df_pain_points, "pain_points")
df_expectations = filter_short(df_expectations, "expectations")
print(f"Dropped ≤{MIN_WORDS}-word rows: pain {before_short[0]} → {len(df_pain_points)}, "
      f"exp {before_short[1]} → {len(df_expectations)}")

df_pain_points.to_csv(CLEAN_CHALLENGES_CSV, index=False, encoding="utf-8-sig")
df_expectations.to_csv(CLEAN_EXPECTATIONS_CSV, index=False, encoding="utf-8-sig")
print(f"Saved → {CLEAN_CHALLENGES_CSV.name} ({len(df_pain_points)}), "
      f"{CLEAN_EXPECTATIONS_CSV.name} ({len(df_expectations)})")
df_pain_points.sample(min(5, len(df_pain_points)), random_state=42)


## 5. Categorize Challenges & Expectations

Hybrid classifier (priority order):

1. **Title keywords** — strong match on the label before `:`
2. **Body keywords** — weighted keyword hits in the full text
3. **Semantic similarity** — cosine match to theme descriptions (`all-mpnet-base-v2`)
4. Weak keyword / semantic fallbacks

Challenges are then clustered with **UMAP → HDBSCAN** (small param sweep) so
related pain points share a `Cluster_Label`.

Category embeddings are computed **once** and reused for both datasets.


In [9]:
CATEGORY_CONFIG = {
    "Workflow & Business Processes": {
        "workflow": 3, "process": 2, "manual": 2, "duplicate entry": 4, "duplicate work": 4,
        "re enter": 4, "reenter": 4, "approval": 3, "routing": 3, "queue": 2, "task": 2,
        "automation": 4, "automate": 4, "streamline": 3, "paper": 2, "paperwork": 2,
        "bottleneck": 3, "delay": 2, "inefficient": 3, "spreadsheet": 4, "excel tracking": 5,
        "manual tracking": 5, "payment": 3, "reconciliation": 4, "action": 2,
        "too many actions": 5, "too many clicks": 5, "human error": 5, "returned": 3,
        "correction": 3, "rework": 4, "front counter": 4, "walk in": 3, "fee": 3,
        "bulk": 3, "status gap": 4, "overwhelming": 4, "traffic": 3,
    },
    "Case Management": {
        "case": 3, "case management": 5, "inspection": 3, "inspector": 3, "violation": 3,
        "complaint": 3, "assignment": 2, "owner": 2, "priority": 2, "status": 2,
        "follow up": 3, "resolution": 2, "permit": 2, "citation": 2, "referral": 5,
        "referral tracking": 5, "ticket": 3, "hearing": 3, "appeal": 3, "evidence": 4,
        "applicant": 3, "constituent": 3,
    },
    "Data Management & Visibility": {
        "data": 2, "information": 2, "duplicate data": 4, "missing data": 4,
        "missing information": 4, "visibility": 4, "single source": 5, "history": 2,
        "property history": 4, "case history": 4, "cross department": 3,
        "shared information": 4, "information flow": 4, "inconsistent": 3, "accuracy": 3,
        "fragmented": 5, "fragmentation": 5, "scattered": 5, "spread across": 4,
        "multiple locations": 4, "historical": 3, "legacy": 3, "migration": 4,
        "synchronized": 4, "complete picture": 5, "linked": 4, "unlinked": 4,
    },
    "Records & Document Management": {
        "search": 4, "lookup": 4, "find": 3, "retrieve": 3, "locate": 3, "record": 2,
        "records": 2, "property research": 5, "parcel": 3, "address": 2, "document": 2,
        "attachment": 2, "photo": 2, "archive": 3, "filter": 2, "plan": 3, "survey": 3,
        "viewer": 4, "upload": 3, "pdf": 3, "notes": 3, "cheat sheet": 4,
    },
    "System Integration": {
        "integration": 5, "integrate": 5, "api": 5, "gis": 4, "onbase": 4, "camino": 4,
        "as400": 4, "esri": 4, "sync": 4, "interface": 3, "external system": 5,
        "third party": 4, "multiple systems": 5, "system switching": 5,
        "too many systems": 5, "too many applications": 5, "aims": 4, "etax": 4,
        "invoice cloud": 4, "word": 3, "outlook": 3, "foxit": 3,
    },
    "Reporting & Decision Support": {
        "report": 4, "reporting": 4, "dashboard": 5, "analytics": 4, "metric": 3,
        "kpi": 4, "statistics": 3, "summary": 2, "export": 3, "excel": 2, "csv": 2,
    },
    "Communication & Collaboration": {
        "communication": 4, "communicate": 4, "notification": 4, "notify": 3, "email": 3,
        "alert": 3, "coordination": 4, "coordinate": 4, "comment": 2, "discussion": 2,
        "collaboration": 4, "handoff": 3, "shared": 2, "call": 4, "phone": 4,
        "disconnect": 5, "call volume": 5, "repeat calls": 5, "customer calls": 4,
        "department": 4, "interdepartmental": 4,
    },
    "Scheduling & Resource Management": {
        "schedule": 4, "scheduling": 4, "calendar": 3, "appointment": 3, "availability": 3,
        "dispatch": 3, "route": 3, "deadline": 2, "timeline": 2, "workload": 3,
        "reschedule": 3, "tickler": 5, "tickle": 5, "reminder": 5, "due date": 4,
        "seasonal": 3,
    },
    "User Experience & Performance": {
        "slow": 4, "performance": 4, "lag": 4, "freeze": 4, "crash": 4, "timeout": 4,
        "loading": 3, "usability": 4, "user friendly": 5, "navigation": 3,
        "click": 2, "screen": 2, "confusing": 3, "easy": 2, "difficult": 2, "mobile": 4,
        "tablet": 4, "desktop": 3, "internet": 4, "offline": 5, "network": 4,
        "connectivity": 4, "shutdown": 5, "restart": 5, "reboot": 5, "exception": 5,
        "memory": 5, "bug": 4, "tab": 3, "large plans": 4, "zoom": 3,
        "display": 3, "module": 4, "features": 1, "dropdown": 3, "action button": 4,
        "cluttered": 4, "organized": 3,
    },
    "Training & Documentation": {
        "training": 5, "documentation": 5, "guide": 3, "instruction": 3,
        "support": 2, "knowledge": 4, "faq": 3, "onboarding": 4, "reference": 2, "sop": 5,
    },
}

CATEGORY_DESCRIPTIONS = {
    cat: f"{cat}. " + ", ".join(sorted(kws, key=lambda k: -kws[k])[:10])
    for cat, kws in CATEGORY_CONFIG.items()
}
CATEGORY_NAMES = list(CATEGORY_DESCRIPTIONS.keys())
print(f"{len(CATEGORY_CONFIG)} themes ready")


10 themes ready


In [10]:
def extract_title(text: str) -> str:
    text = str(text)
    return text.split(":")[0].strip() if ":" in text else text.strip()


def assign_keyword(text: str, category_config: dict) -> tuple[str, float]:
    text_lower = str(text).lower()
    best_category, best_score = "Other", 0.0
    for category, keywords in category_config.items():
        score = sum(w for kw, w in keywords.items() if kw in text_lower)
        if score > best_score:
            best_category, best_score = category, float(score)
    return best_category, best_score


def pick_category(row: pd.Series) -> tuple[str, str, float]:
    if row["title_score"] >= TITLE_KEYWORD_MIN:
        return row["title_category"], "title_keyword", row["title_score"]
    if row["keyword_score"] >= BODY_KEYWORD_MIN:
        return row["keyword_category"], "keyword", row["keyword_score"]
    if row["semantic_score"] >= SEMANTIC_MIN:
        return row["semantic_category"], "semantic", row["semantic_score"]
    if row["keyword_score"] >= BODY_KEYWORD_LOW_MIN:
        return row["keyword_category"], "keyword_low", row["keyword_score"]
    return row["semantic_category"], "semantic_fallback", row["semantic_score"]


def categorize_dataframe(
    frame: pd.DataFrame,
    text_column: str,
    category_config: dict,
    model: SentenceTransformer,
    cat_embeddings: np.ndarray,
) -> tuple[pd.DataFrame, np.ndarray]:
    """Hybrid categorize; returns (dataframe, text_embeddings)."""
    work = frame.copy()
    work["title"] = work[text_column].map(extract_title)
    work["categorize_text"] = work["title"] + ". " + work[text_column].astype(str)

    text_embeddings = model.encode(
        work["categorize_text"].tolist(),
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=64,
    )
    sims = cosine_similarity(text_embeddings, cat_embeddings)
    work["semantic_category"] = [CATEGORY_NAMES[i] for i in sims.argmax(axis=1)]
    work["semantic_score"] = sims.max(axis=1)

    kw = work[text_column].map(lambda v: assign_keyword(v, category_config))
    work["keyword_category"] = kw.map(lambda r: r[0])
    work["keyword_score"] = kw.map(lambda r: r[1])

    title_kw = work["title"].map(lambda v: assign_keyword(v, category_config))
    work["title_category"] = title_kw.map(lambda r: r[0])
    work["title_score"] = title_kw.map(lambda r: r[1])

    picked = work.apply(pick_category, axis=1, result_type="expand")
    work["Category"] = picked[0]
    work["Category_Method"] = picked[1]
    work["Category_Confidence"] = picked[2]
    return work, text_embeddings


# Prefer cleaned frames from Section 4
df = df_pain_points.copy() if "df_pain_points" in dir() else pd.read_csv(CLEAN_CHALLENGES_CSV)
expectations_df = (
    df_expectations.copy() if "df_expectations" in dir() else pd.read_csv(CLEAN_EXPECTATIONS_CSV)
)
df = df.dropna(subset=[CHALLENGE_TEXT_COL]).reset_index(drop=True)
expectations_df = expectations_df.dropna(subset=[EXPECTATION_TEXT_COL]).reset_index(drop=True)

embedder = SentenceTransformer(EMBEDDING_MODEL)
cat_embeddings = embedder.encode(
    list(CATEGORY_DESCRIPTIONS.values()),
    normalize_embeddings=True,
    show_progress_bar=False,
)

df, challenge_embeddings = categorize_dataframe(
    df, CHALLENGE_TEXT_COL, CATEGORY_CONFIG, embedder, cat_embeddings
)
expectations_df, _ = categorize_dataframe(
    expectations_df, EXPECTATION_TEXT_COL, CATEGORY_CONFIG, embedder, cat_embeddings
)

print("Challenge categories:\n", df["Category"].value_counts().to_string())
print("\nMethod breakdown:\n", df["Category_Method"].value_counts().to_string())
low = df[df["Category_Confidence"] < 3].sort_values("Category_Confidence")
print(f"\nLow-confidence rows to spot-check: {len(low)}")
low[["focus_group", "title", "Category", "Category_Method", "Category_Confidence"]].head(10)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Challenge categories:
 Category
Case Management                     48
Records & Document Management       37
User Experience & Performance       33
Workflow & Business Processes       31
System Integration                  30
Communication & Collaboration       20
Scheduling & Resource Management    11
Data Management & Visibility        10
Training & Documentation             6
Reporting & Decision Support         3

Method breakdown:
 Category_Method
title_keyword        120
keyword_low           45
semantic_fallback     34
semantic              18
keyword               12

Low-confidence rows to spot-check: 62


,focus_group,title,Category,Category_Method,Category_Confidence
91,DOCE Building Inspectors,BAS said nothing they can do about it,Case Management,semantic_fallback,0.087267
54,BAA Supervisors & Admin Aide,City won’t pay for Adobe,System Integration,semantic_fallback,0.097316
221,DOCE CPO Coordinators,New cert just stopped working,Training & Documentation,semantic_fallback,0.108019
27,NBD Data Team,Unable to add things to IPS through code | Add...,System Integration,semantic_fallback,0.108661
81,DOCE Building Inspectors,Ashlie - she didn’t know they were – can't cit...,Scheduling & Resource Management,semantic_fallback,0.146413
97,DOCE CPO Coordinators,Special names like MLK spelled incorrectly,Case Management,semantic_fallback,0.159080
165,DOCE Supervisors,"If one mistake made, lost in the abyss",Workflow & Business Processes,semantic_fallback,0.161638
190,IT,Too much outside interference that could break...,Communication & Collaboration,semantic_fallback,0.162602
90,DOCE Building Inspectors,If you try to delete causes issues,Workflow & Business Processes,semantic_fallback,0.181030
222,DOCE CPO Coordinators,Things break and can’t figure out what happened,Workflow & Business Processes,semantic_fallback,0.184261


In [11]:
def assign_cluster_names(
    frame: pd.DataFrame,
    cluster_col: str = "Cluster",
    category_col: str = "Category",
    output_col: str = "Cluster_Label",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Name each cluster after its most common category label."""
    cluster_summary = (
        frame.groupby([cluster_col, category_col]).size().reset_index(name="Count")
    )
    dominant = (
        cluster_summary.sort_values("Count", ascending=False)
        .drop_duplicates(subset=[cluster_col])
        [[cluster_col, category_col]]
        .rename(columns={category_col: output_col})
    )
    return frame.merge(dominant, on=cluster_col, how="left"), cluster_summary


TARGET_CLUSTERS = len(CATEGORY_CONFIG)

# UMAP once, then a compact HDBSCAN sweep (mcs 6–12 keeps quality without a huge grid)
reducer = umap.UMAP(
    n_neighbors=20, n_components=15, min_dist=0.0, metric="cosine", random_state=42
)
reduced = reducer.fit_transform(challenge_embeddings)

candidate_params = [
    (mcs, ms)
    for mcs in range(6, 13)
    for ms in sorted({1, 2, max(2, mcs // 2)})
]

sweep_rows, best_labels, best_params, best_score = [], None, None, -np.inf
for mcs, ms in candidate_params:
    labels = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric="euclidean",
        cluster_selection_method="eom",
    ).fit_predict(reduced)

    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    noise = int((labels == -1).sum())
    silhouette = (
        silhouette_score(reduced[mask], labels[mask], metric="euclidean")
        if mask.sum() > 1 and n_clusters > 1
        else -1.0
    )
    combined = silhouette - abs(n_clusters - TARGET_CLUSTERS) * 0.02 - noise / len(df) * 0.5
    sweep_rows.append({
        "min_cluster_size": mcs, "min_samples": ms, "clusters": n_clusters,
        "noise": noise, "silhouette": round(float(silhouette), 4),
        "combined_score": round(float(combined), 4),
    })
    if combined > best_score:
        best_score, best_labels, best_params = combined, labels, (mcs, ms)

display(
    pd.DataFrame(sweep_rows)
    .sort_values(["combined_score", "silhouette"], ascending=False)
    .head(8)
)
print(f"Chosen: mcs={best_params[0]}, ms={best_params[1]}, score={best_score:.4f}")

df["Cluster"] = best_labels
df, cluster_summary = assign_cluster_names(df)
print(f"Clusters: {(df['Cluster'] != -1).sum() and df.loc[df['Cluster'] != -1, 'Cluster'].nunique()} "
      f"| noise: {(df['Cluster'] == -1).sum()}")
print(df["Cluster_Label"].value_counts().to_string())


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,min_cluster_size,min_samples,clusters,noise,silhouette,combined_score
5,7,3,10,57,0.5603,0.4359
3,7,1,10,22,0.4653,0.4173
6,8,1,10,22,0.4653,0.4173
2,6,3,11,51,0.5212,0.3898
0,6,1,11,22,0.4543,0.3863
1,6,2,11,28,0.4668,0.3857
9,9,1,9,21,0.4359,0.3701
8,8,4,8,61,0.5396,0.3664


Chosen: mcs=7, ms=3, score=0.4359
Clusters: 10 | noise: 57
Cluster_Label
Case Management                  106
Records & Document Management     41
User Experience & Performance     32
System Integration                30
Workflow & Business Processes     12
Communication & Collaboration      8


In [12]:
# Persist categorized outputs
challenge_cols = [
    c for c in [
        "focus_group", CHALLENGE_TEXT_COL, "processed_text", "sentiment",
        "title", "Category", "Category_Method", "Category_Confidence",
        "Cluster", "Cluster_Label",
    ] if c in df.columns
]
expectation_cols = [
    c for c in [
        "focus_group", EXPECTATION_TEXT_COL, "processed_text", "sentiment",
        "title", "Category", "Category_Method", "Category_Confidence",
    ] if c in expectations_df.columns
]

df[challenge_cols].to_csv(CATEGORIZED_CHALLENGES_CSV, index=False, encoding="utf-8-sig")
expectations_df[expectation_cols].to_csv(
    CATEGORIZED_EXPECTATIONS_CSV, index=False, encoding="utf-8-sig"
)

category_summary = (
    df.groupby("Category", as_index=False).size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=False)
)
category_summary.to_csv(CATEGORY_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print(f"Saved {CATEGORIZED_CHALLENGES_CSV.name} ({len(df)} rows)")
print(f"Saved {CATEGORIZED_EXPECTATIONS_CSV.name} ({len(expectations_df)} rows)")
print(f"Saved {CATEGORY_SUMMARY_CSV.name}")
display(category_summary)

# Quick cluster sample
for cluster in sorted(df["Cluster"].unique())[:5]:
    label = df.loc[df["Cluster"] == cluster, "Cluster_Label"].iloc[0]
    print("=" * 72)
    print(f"Cluster {cluster} → {label}")
    for text in df.loc[df["Cluster"] == cluster, CHALLENGE_TEXT_COL].head(2):
        print("-", text)


Saved categorized_challenges.csv (229 rows)
Saved categorized_expectations.csv (47 rows)
Saved category_summary.csv


,Category,Count
0,Case Management,48
3,Records & Document Management,37
8,User Experience & Performance,33
9,Workflow & Business Processes,31
6,System Integration,30
1,Communication & Collaboration,20
5,Scheduling & Resource Management,11
2,Data Management & Visibility,10
7,Training & Documentation,6
4,Reporting & Decision Support,3


Cluster -1 → Case Management
- Too many interfaces not linking info (i.e. parcel search)
- 60day notices: crashes and takes time to process | Migrate list (tracking) out of IPS (this can be an excel file)
Cluster 0 → System Integration
- Application process made more difficult with Camino | Get rid of Camino!
- Clunky – easy to accidentally break things inaditantly – make codes item, had to regenerate codes book with rendor
Cluster 1 → Communication & Collaboration
- OnBase layout and windows often not very user-friendly
- Our phones are on a time limit – 5min max – drops automatically – IT has known about this issue – people call back and yell because they think they hung up
Cluster 2 → Case Management
- Getting blamed for slow BAA and Law process
- Too many actions | Eliminate unneeded actions i.e. (BAA Ticket Rounds)
Cluster 3 → Records & Document Management
- I can’t complete an affirmation if the notices and nail & mail aren’t uploaded into IPS
- Constituents can’t access photos c

## 6. Visualize

Charts reuse the **in-memory** categorized frame and challenge embeddings from
Section 5 (no second model load). If you restarted the kernel, re-run Section 5
first — or load the CSV and re-embed below as a fallback.


In [13]:
# Ensure df + embeddings exist (fallback if kernel was restarted after Section 5)
if "df" not in dir() or "challenge_embeddings" not in dir():
    df = pd.read_csv(CATEGORIZED_CHALLENGES_CSV)
    if "embedder" not in dir():
        embedder = SentenceTransformer(EMBEDDING_MODEL)
    categorize_text = df["title"] + ". " + df[CHALLENGE_TEXT_COL].astype(str)
    challenge_embeddings = embedder.encode(
        categorize_text.tolist(),
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=64,
    )

# Category counts
category_counts = (
    df.groupby("Category", as_index=False).size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_cat = px.bar(
    category_counts, x="Count", y="Category", orientation="h",
    title="Pain Point Categories (Hybrid Classifier)",
    color="Count", color_continuous_scale="Blues",
)
fig_cat.update_layout(showlegend=False, height=500)
fig_cat.show()

# Cluster sizes
cluster_counts = (
    df.groupby(["Cluster", "Cluster_Label"], as_index=False).size()
    .rename(columns={"size": "Count"})
    .sort_values("Cluster")
)
fig_clusters = px.bar(
    cluster_counts, x="Count", y="Cluster_Label", color="Cluster_Label",
    orientation="h", title="Semantic Clusters (HDBSCAN)",
)
fig_clusters.update_layout(showlegend=False, height=500)
fig_clusters.show()

# 2D semantic map
coords_2d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(
    challenge_embeddings
)
plot_df = pd.DataFrame({
    "x": coords_2d[:, 0],
    "y": coords_2d[:, 1],
    "Category": df["Category"],
    "Cluster_Label": df["Cluster_Label"],
    "Challenge": df[CHALLENGE_TEXT_COL],
    "Focus Group": df["focus_group"],
})
fig_map = px.scatter(
    plot_df, x="x", y="y", color="Category",
    hover_data=["Focus Group", "Cluster_Label", "Challenge"],
    title="2D Semantic Map by Theme",
)
fig_map.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# Focus Group × Category heatmap (click a cell when widgets are available)
heatmap_df = pd.crosstab(df["focus_group"], df["Category"])
fg_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
cat_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[fg_order, cat_order].T.loc[cat_order, fg_order]

fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Focus Group", y="Category", color="Count"),
    title="Focus Group vs Category (click a cell to view records)",
    color_continuous_scale="YlOrRd",
    text_auto=True,
    aspect="auto",
)
fig_heatmap.update_traces(textfont=dict(size=11))
fig_heatmap.update_layout(
    height=700, width=1100, xaxis_tickangle=-45,
    coloraxis_colorbar=dict(tickmode="linear", dtick=2),
)

if HAS_WIDGETS:
    fig = go.FigureWidget(fig_heatmap)
    output = widgets.Output()

    def show_records(trace, points, selector):
        if not points.point_inds:
            return
        focus_group, category = points.xs[0], points.ys[0]
        selected = df[
            (df["focus_group"] == focus_group) & (df["Category"] == category)
        ][["focus_group", "title", CHALLENGE_TEXT_COL, "sentiment"]].reset_index(drop=True)
        with output:
            output.clear_output(wait=True)
            display(Markdown(f"### {focus_group} · {category}"))
            display(Markdown(f"**{len(selected)} record(s)**"))
            display(selected)

    fig.data[0].on_click(show_records)
    display(widgets.VBox([fig, output]))
else:
    fig_heatmap.show()
